In [6]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

## Load Data

In [8]:
from app.data.ingest import load_transactions

df = load_transactions("../data/synthetic/transactions.csv")
df.head()

,transaction_id,customer_id,transaction_amount,transaction_hour,merchant_category,customer_tenure_days,num_transactions_24h,num_failed_transactions_24h,avg_transaction_amount_30d,is_foreign_transaction,device_type,country,previous_chargebacks,account_age_days,risk_score_external,is_fraud
0,1,6818,137.15,15,fashion,1847,1,1,140.84,False,mobile,US,2,2199,0.0007,0
1,2,2161,21.63,22,grocery,2954,1,0,42.36,False,desktop,US,0,2981,0.0000,0
2,3,2278,19.15,22,fashion,188,0,1,18.21,False,mobile,GB,0,261,0.4126,0
3,4,8347,54.76,2,entertainment,925,4,1,36.12,False,pos,GB,0,1030,0.6808,0
4,5,4153,59.96,21,utilities,3329,0,0,78.17,False,pos,IN,0,3654,0.0000,0


## Validation

In [9]:
from app.data.validation import run_data_validation

validation_report = run_data_validation(df)
validation_report

{'passed': True,
 'num_rows': 50000,
 'num_columns': 16,
 'columns': ['transaction_id',
  'customer_id',
  'transaction_amount',
  'transaction_hour',
  'merchant_category',
  'customer_tenure_days',
  'num_transactions_24h',
  'num_failed_transactions_24h',
  'avg_transaction_amount_30d',
  'is_foreign_transaction',
  'device_type',
  'country',
  'previous_chargebacks',
  'account_age_days',
  'risk_score_external',
  'is_fraud'],
 'errors': [],
 'warnings': ['Column transaction_amount contains extreme high outliers. 99th percentile=763.91, 99.9th percentile=2385.83'],
 'target_counts': {'0': 48469, '1': 1531},
 'positive_count': 1531,
 'negative_count': 48469,
 'fraud_rate': 0.03062}

## Profile

In [10]:
from app.data.profiling import  generate_data_profile

profile = generate_data_profile(df)
profile['fraud_rate'], profile['majority_class_accuracy']

(0.03062, 0.96938)

## Target distribution

In [11]:
df['is_fraud'].value_counts(normalize=True)

is_fraud
0    0.96938
1    0.03062
Name: proportion, dtype: float64

## Numerical distributions

In [12]:
df[['transaction_amount', 'risk_score_external', 'num_transactions_24h']].describe()

,transaction_amount,risk_score_external,num_transactions_24h
count,50000.000000,50000.000000,50000.000000
mean,93.598724,0.091101,1.496480
std,228.274824,0.193763,1.219745
min,1.000000,0.000000,0.000000
25%,24.330000,0.000100,1.000000
50%,48.040000,0.002500,1.000000
75%,96.752500,0.061900,2.000000
max,19148.900000,0.997900,10.000000


## Fraud rate by merchant category

In [14]:
df.groupby('merchant_category')['is_fraud'].mean().sort_values(ascending=False)

merchant_category
gambling         0.052862
electronics      0.046418
crypto           0.034460
entertainment    0.030934
utilities        0.027913
fashion          0.027857
restaurant       0.026288
other            0.026073
grocery          0.024790
travel           0.024566
Name: is_fraud, dtype: float64

## Modeling implications

- The dataset is highly imbalanced.
- Accuracy is not useful as a primary metric.
- PR-AUC, recall, precision, fraud capture rate, and expected cost should be prioritized.
- Threshold tuning will be important because the business cost of false negatives is much higher than false positives.